In [0]:
from datetime import datetime
import uuid

run_id = "uc16_" + datetime.utcnow().strftime("%Y%m%d_%H%M%S")
task_name = "build_appointments_clean"

spark.sql(f"""
INSERT INTO poc_health_no_show.ops.pipeline_run_audit
VALUES (
  '{run_id}',
  'uc16_no_show_daily_pipeline',
  '{task_name}',
  'STARTED',
  current_timestamp(),
  NULL,
  NULL,
  NULL
)
""")

In [0]:
%sql
CREATE OR REPLACE TABLE poc_health_no_show.gold.outreach_tracking_simulated
COMMENT 'Clearly labelled simulated A/B outreach outcomes for demonstration only.'
AS
SELECT
  uuid() AS tracking_id,
  v.appointment_id,
  v.variant,
  'SMS' AS channel,
  current_timestamp() AS sent_at,
  'SIMULATED_SENT' AS delivery_status,
  CASE
    WHEN rand() < 0.35 THEN 'CONFIRMED'
    WHEN rand() < 0.65 THEN 'NO_RESPONSE'
    ELSE 'RESCHEDULE_REQUESTED'
  END AS confirmation_status,
  CASE
    WHEN rand() < 0.75 THEN TRUE
    ELSE FALSE
  END AS attended_flag,
  current_timestamp() AS created_at
FROM poc_health_no_show.gold.outreach_message_variants v;

In [0]:
%sql
SELECT
  variant,
  COUNT(*) AS outreach_count,
  ROUND(AVG(CASE WHEN attended_flag THEN 1.0 ELSE 0.0 END) * 100, 2) AS simulated_attendance_rate_pct
FROM poc_health_no_show.gold.outreach_tracking_simulated
GROUP BY variant;

In [0]:
spark.sql(f"""
INSERT INTO poc_health_no_show.ops.pipeline_run_audit
VALUES (
  '{run_id}',
  'uc16_no_show_daily_pipeline',
  '{task_name}',
  'SUCCESS',
  current_timestamp(),
  current_timestamp(),
  NULL,
  NULL
)
""")